# RR BENCHMARK PHASE B2: the scoring run, n = 7
Scores the FROZEN signatures and drug IDs from `signatures_frozen_B1b.json` under the committed pre-registration (SHA-256 7eee1779...c9a2) and addenda v1.1, v1.2, v1.3.
**Criterion: one condition, permutation p < 0.05 on the mean percentile rank of the approved drugs, 10,000 permutations.** Lower percentile = stronger reversal. Ties take the midrank.
Run only AFTER B1b has finished. Run: hamburger > Runtime > Run all. Paste back everything between `=== PHASE B2 REPORT BEGIN ===` and `=== PHASE B2 REPORT END ===`.

In [ ]:
# CELL 1: load frozen inputs; nothing below may change them
from google.colab import drive
drive.mount("/content/drive")
import os,json,hashlib,time,numpy as np,pandas as pd
W="/content/drive/MyDrive/RitschelResearch/benchmark"; os.chdir(W)
raw=open("signatures_frozen_B1b.json","rb").read(); FROZEN_SHA=hashlib.sha256(raw).hexdigest()
FZ=json.loads(raw); SIG=FZ["signatures"]; DR=pd.DataFrame(FZ["drugs"])
ROWS=["6","8","10","12","14/15","17","18"]
assert sorted(SIG)==sorted(ROWS), ("ROW SET IS NOT THE PRE-REGISTERED SEVEN",sorted(SIG))
DRUGROW={"6":["6"],"8":["8"],"10":["10"],"12":["12"],"14/15":["14"],"17":["17"],"18":["18"]}
M=np.load("libA_matrix.npy"); perts=np.load("libA_perts.npy",allow_pickle=True); genes=np.load("libA_genes.npy")
GI={int(x):i for i,x in enumerate(genes)}
assert M.shape==(90834,978), M.shape
assert FROZEN_SHA=="1457464f13bac9a7bdeca438f44414f4a2c2a7e0562992375b982217fd5c54e9", "NOT THE CHECKED B1b FREEZE"
print("frozen sha256",FROZEN_SHA,"library",M.shape)

In [ ]:
# CELL 1b: GUARD before any score. Row 6: every GSM that contributed to a donor mean must have spots.
# (29 GSMs had zero library size and 29 had no donor; this proves they are the same 29.)
import tarfile,gzip
meta=[]
for line in open("GSE174647_series_matrix.txt",encoding="utf-8",errors="replace"):
    if line.startswith("!series_matrix_table_begin"): break
    meta.append(line)
title=[x.strip().strip('"') for x in [l for l in meta if l.startswith("!Sample_title")][0].rstrip("\n").split("\t")[1:]]
sp=pd.read_csv("GSE174647_ST_spatial_coord_annotations_spots.csv.gz",usecols=["id","donor_pseud"]).drop_duplicates()
with_donor=set(t for t in title if t in set(sp.id))
with tarfile.open("GSE174647_ST_counts.tar.gz") as tf:
    bc=pd.read_csv(tf.extractfile("ST_counts/barcodes.tsv"),sep="\t",header=None)[0].astype(str)
with_spots=set(b.rsplit("_",1)[0] for b in bc)&set(title)
print("row 6: GSMs with donor",len(with_donor),"| with spots",len(with_spots),"| donor but NO spots",len(with_donor-with_spots))
assert with_donor<=with_spots, "ROW 6 DEFECT: a zero-spot GSM entered a donor mean. STOP, do not score."
print("GUARD PASSED: no zero-spot GSM entered any donor mean")

In [ ]:
# CELL 2: control-2 scoring code, unchanged (Lamb KS connectivity; 0 when up and down enrich the same way)
def ks(rp,n,t):
    if t==0 or n==0: return 0.0
    V=np.sort(np.asarray(rp,float)); j=np.arange(1,t+1,dtype=float)
    a=np.max(j/t-V/n); b=np.max(V/n-(j-1.0)/t)
    return float(a) if a>b else float(-b)
def conn_all(Mx,up,dn):
    n=Mx.shape[1]; out=np.empty(Mx.shape[0])
    order=np.argsort(-Mx,axis=1,kind="stable")
    rank=np.empty_like(order); ar=np.arange(1,n+1)
    np.put_along_axis(rank,order,np.broadcast_to(ar,order.shape),axis=1)
    for i in range(Mx.shape[0]):
        eu=ks(rank[i,up],n,len(up)); ed=ks(rank[i,dn],n,len(dn))
        out[i]=0.0 if eu*ed>=0 else (eu-ed)/2.0
    return out
def midrank(v):
    v=np.asarray(v,float); n=v.size; o=np.argsort(v,kind="stable"); r=np.empty(n); i=0
    while i<n:
        j=i
        while j+1<n and v[o[j+1]]==v[o[i]]: j+=1
        r[o[i:j+1]]=0.5*(i+j)+1.0; i=j+1
    return r

In [ ]:
# CELL 3: score every row. Compound score = median over its signatures (pre-registered). Percentile = (midrank-0.5)/N, ascending.
PCT={}; TIE={}; NCMP={}; t0=time.time()
for row in ROWS:
    s=SIG[row]; cols=[GI[e] for e in s["usable"]]; pos={e:i for i,e in enumerate(s["usable"])}
    up=np.array([pos[e] for e in s["up"]]); dn=np.array([pos[e] for e in s["dn"]])
    assert len(up)==150 and len(dn)==150 and not set(up)&set(dn)
    sc=conn_all(M[:,cols],up,dn)
    comp=pd.DataFrame({"p":perts,"s":sc}).groupby("p")["s"].median()
    pct=pd.Series((midrank(comp.values)-0.5)/len(comp),index=comp.index)
    PCT[row]=pct; TIE[row]=float((comp==0).mean()); NCMP[row]=len(comp)
    print(f"row {row}: compounds {len(comp)} exact-zero {TIE[row]:.3f} {round(time.time()-t0)}s",flush=True)

In [ ]:
# CELL 4: observed statistic. Drug = median over its IDs (v1.1); row = median over its drugs (myocardium has three).
def row_score(row,pct,draw=None):
    drugs=DR[DR.row.astype(str).isin(DRUGROW[row])]
    per=[]
    for d,g in drugs.groupby("drug"):
        ids=list(g.pert_id) if draw is None else draw[d]
        per.append(float(np.median(pct.reindex(ids).values)))
    return float(np.median(per)),per
OBS={}; DETAIL=[]
for row in ROWS:
    pct=PCT[row]; drugs=DR[DR.row.astype(str).isin(DRUGROW[row])]
    missing=[p for p in drugs.pert_id if p not in pct.index]; assert not missing, ("DRUG NOT SCORED",row,missing)
    OBS[row],per=row_score(row,pct)
    for d,g in drugs.groupby("drug"):
        for p in g.pert_id: DETAIL.append((row,d,p,float(pct[p]),float(pct[p])!=float(pct.median()) ))
STAT=float(np.mean([OBS[r] for r in ROWS]))
print("observed mean percentile",round(STAT,4))

In [ ]:
# CELL 5: permutation null, 10,000 draws. Each row: every drug's k IDs replaced by k distinct random library compounds
# (distinct across the row's drugs), drawn independently per row. Same median structure recomputed each time.
B=10000; rng=np.random.default_rng(20261001); NULL=np.empty(B); t0=time.time()
STRUCT={row:[(d,len(g)) for d,g in DR[DR.row.astype(str).isin(DRUGROW[row])].groupby("drug")] for row in ROWS}
ARR={row:(PCT[row].values) for row in ROWS}
for b in range(B):
    tot=0.0
    for row in ROWS:
        need=sum(k for d,k in STRUCT[row]); pick=rng.choice(len(ARR[row]),size=need,replace=False); i=0; per=[]
        for d,k in STRUCT[row]:
            per.append(np.median(ARR[row][pick[i:i+k]])); i+=k
        tot+=np.median(per)
    NULL[b]=tot/len(ROWS)
p=(1+int((NULL<=STAT).sum()))/(B+1)
print("null mean",round(NULL.mean(),4),"sd",round(NULL.std(),4),"p",p,round(time.time()-t0),"s")

In [ ]:
# CELL 6: report
out=dict(frozen_sha=FROZEN_SHA,rows=ROWS,observed=STAT,p=p,B=B,null_mean=float(NULL.mean()),null_sd=float(NULL.std()),
         per_row={r:OBS[r] for r in ROWS},tie_frac=TIE,n_compounds=NCMP,detail=DETAIL)
b=json.dumps(out,sort_keys=True).encode(); open("B2_result.json","wb").write(b)
assert open("B2_result.json","rb").read()==b
np.save("B2_null.npy",NULL)
print("=== PHASE B2 REPORT BEGIN ===")
print("scored against signatures_frozen_B1b.json sha256",FROZEN_SHA)
print("rows",ROWS)
print("row | percentile (lower = stronger reversal) | library exact-zero fraction | compounds")
for r in ROWS: print(f"  {r:6s} {OBS[r]:.4f}   tie {TIE[r]:.3f}   n {NCMP[r]}")
print("drug ID detail (row, drug, pert_id, percentile):")
for row,d,pid,v,_ in DETAIL: print(f"  {row:6s} {d:15s} {pid} {v:.4f}")
print(f"OBSERVED MEAN PERCENTILE {STAT:.4f}")
print(f"NULL mean {NULL.mean():.4f} sd {NULL.std():.4f}  ({B} permutations)")
print(f"PERMUTATION p = {p:.5f}")
print("CRITERION (p < 0.05):", "MET" if p<0.05 else "NOT MET")
print("B2_result.json sha256",hashlib.sha256(b).hexdigest())
print("=== PHASE B2 REPORT END ===")